# Этап 4 — небольшой контролируемый пилот PPO
**Подготовлено, но этот этап при разработке не выполнялся в Colab.**
По умолчанию `RUN_TESTS=False`, `ENABLE_TRAINING=False`: только установка/preflight,
просмотр замороженного плана и сохранение provenance. Реальные данные и модели не загружаются.

План: 3 общих seed × (1 none + 4 fixed + 4 adaptive) = **27 запусков по 16384 шага**,
всего **442368 train-шагов**. Fixed/adaptive: по 12 запусков / 196608 шагов.
Это разведочный ограниченный поиск; test 2022–2025 остаётся закрытым.
Сначала прочитайте `docs/PILOT_PROTOCOL.md` и короткий `docs/COLAB_QUICKSTART.md`.

## 1. GitHub и явные переключатели
Откройте этот notebook из опубликованного Stage 4 commit. В `GIT_REF` задайте его полный SHA.
`RUN_TESTS=True` запускает полный pytest (включает крошечные **синтетические** обучения).
Реальные 27 запусков требуют отдельно `ENABLE_TRAINING=True`; при этом RUN_TESTS должен быть True.
CPU используется по умолчанию; CUDA выбирается явно и не подменяется CPU при ошибке.

Новый BATCH_ID оставьте пустым. Для продолжения после перезапуска укажите прежний BATCH_ID,
тот же SHA/plan/device. Если осталась блокировка оборванного runtime, сначала удалите старый
runtime и только затем явно установите RECOVER_INTERRUPTED=True. Одновременная работа двух
ноутбуков с одной batch запрещена. Checkpoints позволяют загрузить policy; прерванный job
перезапускается целиком, а не продолжает битовую траекторию optimizer/RNG.

In [ ]:
import sys
import os
import re
import json
import subprocess
import tempfile
from pathlib import Path

try:
    import google.colab
except ImportError as error:
    raise RuntimeError("Запуск разрешён только в Google Colab, не в локальном runtime") from error
if not Path("/content").is_dir() or "COLAB_RELEASE_TAG" not in os.environ:
    raise RuntimeError("Нужна управляемая сессия Google Colab с каталогом /content")
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError("Нужен Python 3.11/3.12. Выберите Runtime > Change runtime type > Runtime Version 2026.07 (Python 3.12).")

GITHUB_REPOSITORY = "Go9shi/Science_Paper"
GIT_REF = "main"  # Replace with the full published Stage 4 evaluator/training SHA.
USE_GITHUB_SECRET = False
CHECK_RL_COMPATIBILITY = True
RUN_TESTS = False  # Explicit opt-in; includes tiny synthetic PPO training tests.
ENABLE_TRAINING = False  # Explicit opt-in for the real, frozen 27-run pilot.
RECOVER_INTERRUPTED = False  # Only after ending the previous runtime if its writer lock remains.
DEVICE = "cpu"  # Optional "cuda"; use the same device/runtime when resuming a batch.
BATCH_ID = ""  # Blank creates a new batch; use the printed ID to resume.
SOURCE_RUN_ID = "20261107T154124Z-9f1f7c1e"
NOTEBOOK_REVISION = "controlled-pilot-v1"
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
if any(type(flag) is not bool for flag in (RUN_TESTS, ENABLE_TRAINING, RECOVER_INTERRUPTED)):
    raise ValueError("Execution switches must be Python booleans")
if ENABLE_TRAINING and not RUN_TESTS:
    raise ValueError("Real training requires RUN_TESTS=True in this attempt")
if DEVICE not in {"cpu", "cuda"} or (BATCH_ID and not re.fullmatch(r"[A-Za-z0-9_-]+", BATCH_ID)):
    raise ValueError("Invalid device or BATCH_ID")
REPO_URL = f"https://github.com/{GITHUB_REPOSITORY}.git"
if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", GITHUB_REPOSITORY):
    raise ValueError("Ожидается repository в формате owner/name")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._/-]*", GIT_REF):
    raise ValueError("Некорректный GIT_REF")
if not re.fullmatch(r"[A-Za-z0-9_-]+", SOURCE_RUN_ID):
    raise ValueError("Некорректный SOURCE_RUN_ID")

REPO_ROOT = Path(tempfile.mkdtemp(prefix="Science_Paper-stage4-", dir="/content"))
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
with tempfile.TemporaryDirectory(prefix="stage1-auth-", dir="/content") as auth_dir:
    if USE_GITHUB_SECRET:
        from google.colab import userdata
        try:
            git_env["STAGE1_GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
        except Exception as error:
            raise RuntimeError("Разрешите доступ к Colab Secret GITHUB_TOKEN с правом чтения репозитория") from error
        if not git_env["STAGE1_GITHUB_TOKEN"]:
            raise RuntimeError("GITHUB_TOKEN пуст")
        askpass = Path(auth_dir) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['STAGE1_GITHUB_TOKEN'])\n",
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        git_env["GIT_ASKPASS"] = str(askpass)
    try:
        subprocess.run(["git", "-c", "credential.helper=", "clone", "--quiet", "--no-checkout",
                        REPO_URL, str(REPO_ROOT)], env=git_env, check=True, capture_output=True, text=True, timeout=180)
        if re.fullmatch(r"[0-9a-fA-F]{7,40}", GIT_REF):
            # Full clone already contains reachable history; resolve short SHAs locally.
            resolved = subprocess.run(
                ["git", "-C", str(REPO_ROOT), "rev-parse", "--verify", f"{GIT_REF}^{{commit}}"],
                capture_output=True, text=True,
            )
            if resolved.returncode == 0:
                SOURCE_COMMIT = resolved.stdout.strip()
            elif len(GIT_REF) < 40:
                raise ValueError("Short SHA was not found or is ambiguous. Use the full 40-character commit SHA.")
            else:
                subprocess.run(["git", "-c", "credential.helper=", "-C", str(REPO_ROOT), "fetch", "--quiet", "origin", GIT_REF],
                               env=git_env, check=True, capture_output=True, text=True, timeout=180)
                SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "FETCH_HEAD"], text=True).strip()
        else:
            subprocess.run(["git", "-c", "credential.helper=", "-C", str(REPO_ROOT), "fetch", "--quiet", "origin", GIT_REF],
                           env=git_env, check=True, capture_output=True, text=True, timeout=180)
            SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "FETCH_HEAD"], text=True).strip()
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--quiet", "--detach", SOURCE_COMMIT], check=True)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as error:
        raise RuntimeError("Не удалось получить код GitHub. Проверьте сеть, GIT_REF и доступ; для private используйте Secret.") from error
    finally:
        git_env.pop("STAGE1_GITHUB_TOKEN", None)
        git_env.pop("GIT_ASKPASS", None)

PROJECT_ROOT = REPO_ROOT / "research"
required = ["AGENTS.md", "src/__init__.py", "src/data.py", "src/volatility.py", "src/validation.py", "src/environment.py",
            "tests/test_data.py", "tests/test_volatility.py", "tests/test_validation.py", "tests/test_environment.py",
            "requirements-colab.txt", "requirements-rl-check.txt", "docs/METHODOLOGY.md", "docs/STAGE1_VALIDATION.md", "docs/COLAB_ENVIRONMENT.md",
            "notebooks/01_data_preparation.ipynb"]
if not CHECK_RL_COMPATIBILITY:
    raise RuntimeError("Stage 4 requires CHECK_RL_COMPATIBILITY=True")
required += ["src/portfolio.py", "src/environment_data.py", "src/portfolio_env.py", "src/env_validation.py",
             "tests/test_portfolio_env.py", "docs/STAGE2_VALIDATION.md", "notebooks/02_environment_validation.ipynb"]
required += ["src/training_config.py", "src/training_data.py", "src/ppo_training.py", "src/financial_metrics.py",
             "src/regularization.py", "src/replay_audit.py", "configs/ppo_smoke.json", "configs/ppo_pilot.json",
             "tests/test_ppo_training.py", "docs/STAGE3_VALIDATION.md", "notebooks/03_ppo_training.ipynb"]
required += ["src/pilot.py", "src/pilot_plan.py", "tests/test_pilot.py", "configs/pilot_plan.json",
             "docs/PILOT_PROTOCOL.md", "docs/COLAB_QUICKSTART.md", "notebooks/04_pilot_experiments.ipynb"]
missing = [name for name in required if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Commit {SOURCE_COMMIT} не содержит актуальный этап 4: {missing}. "
                            "Опубликуйте файлы и выберите содержащий их ref; локальные файлы не подменяют GitHub.")
source_notebook = json.loads((PROJECT_ROOT / "notebooks/04_pilot_experiments.ipynb").read_text(encoding="utf-8"))
if source_notebook["metadata"].get("stage4_revision") != NOTEBOOK_REVISION:
    raise RuntimeError("Версия открытого notebook не соответствует checkout; откройте notebook из выбранного commit")
print("GitHub:", GITHUB_REPOSITORY)
print("Resolved commit:", SOURCE_COMMIT)
print("PROJECT_ROOT:", PROJECT_ROOT)

## 2. Google Drive и проверка записи/чтения

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    raise RuntimeError("Google Drive не подключён: каталог MyDrive недоступен")
OUTPUT_ROOT = DRIVE_ROOT / "ppo_turnover_research" / "stage4"
PREFLIGHT_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
PREFLIGHT_DIR = OUTPUT_ROOT / "preflight" / PREFLIGHT_ID
try:
    PREFLIGHT_DIR.mkdir(parents=True, exist_ok=False)
    probe = PREFLIGHT_DIR / "drive-probe.txt"
    probe_value = uuid4().hex
    probe.write_text(probe_value, encoding="utf-8")
    if probe.read_text(encoding="utf-8") != probe_value:
        raise OSError("Drive read-back mismatch")
    probe.unlink()
except OSError as error:
    raise RuntimeError(f"Нет надёжного чтения/записи Google Drive: {OUTPUT_ROOT}") from error
print("Drive read/write: OK")
print("Preflight logs:", PREFLIGHT_DIR)
STAGE1_RUN = DRIVE_ROOT / "ppo_turnover_research" / "stage1" / "runs" / SOURCE_RUN_ID
if not (STAGE1_RUN / "metadata.json").is_file():
    raise FileNotFoundError(f"Complete Stage 1 directory is unavailable: {STAGE1_RUN}")
print("Stage 1 source:", STAGE1_RUN)

## 3. Окружение Colab env-v2
Runtime 2026.07 / Python 3.12, предустановленный Torch 2.11.0; версии NumPy/pandas и
Gymnasium/SB3 сохраняются. Системные пакеты не обновляются без необходимости.
Выполняются pinned install, imports, NumPy/Torch bridge и `python -m pip check`.
Raw pip check сохраняется: известные Jedi/IPython и Toolz/Ibis warnings допускаются
существующей политикой; новые/активные конфликты блокируют продолжение.

In [ ]:
import importlib.metadata

# Read distribution metadata only: do not import NumPy/Pandas before pip.
def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

before_install = {name: installed_version(name) for name in (
    "numpy", "pandas", "torch", "torchvision", "torchaudio", "google-colab",
    "ipython", "jedi", "ibis-framework", "toolz", "numba", "exchange-calendars",
)}
(PREFLIGHT_DIR / "versions-before-install.json").write_text(
    json.dumps(before_install, indent=2), encoding="utf-8")
install_args = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "--report", str(PREFLIGHT_DIR / "pip-install-report.json"),
                "-r", str(PROJECT_ROOT / "requirements-colab.txt")]
if CHECK_RL_COMPATIBILITY:
    if before_install["torch"] is None:
        raise RuntimeError("В runtime нет PyTorch. Выберите Colab 2026.07; автоматически скачивать/заменять torch не будем.")
    # Keep the exact preinstalled build, including its CUDA/CPU local suffix.
    constraint = PREFLIGHT_DIR / "runtime-constraints.txt"
    constraint.write_text(f"torch=={before_install['torch']}\n", encoding="utf-8")
    install_args += ["-r", str(PROJECT_ROOT / "requirements-rl-check.txt"), "-c", str(constraint)]
install = subprocess.run(install_args, text=True, capture_output=True)
(PREFLIGHT_DIR / "pip-install.log").write_text(install.stdout + "\n" + install.stderr, encoding="utf-8")
if install.returncode != 0:
    print(install.stdout[-4000:])
    print(install.stderr[-4000:])
    raise RuntimeError(f"Установка не прошла: {PREFLIGHT_DIR / 'pip-install.log'}; не обходите конфликт resolver.")
if CHECK_RL_COMPATIBILITY and installed_version("torch") != before_install["torch"]:
    raise RuntimeError("Версия PyTorch изменилась; проверка окружения остановлена")
print("Установка завершена без --upgrade; лог:", PREFLIGHT_DIR / "pip-install.log")

In [ ]:
import importlib
import importlib.metadata
import textwrap

requirements = {}
requirement_files = [PROJECT_ROOT / "requirements-colab.txt"]
if CHECK_RL_COMPATIBILITY:
    requirement_files.append(PROJECT_ROOT / "requirements-rl-check.txt")
for req_file in requirement_files:
    for line in req_file.read_text(encoding="utf-8").splitlines():
        if "==" in line and not line.lstrip().startswith("#"):
            package, version = line.strip().split("==")
            requirements[package] = version
module_names = {"stable-baselines3": "stable_baselines3"}
for package, version in requirements.items():
    installed = importlib.metadata.version(package)
    loaded = getattr(sys.modules.get(module_names.get(package, package)), "__version__", installed)
    if installed != version or loaded != version:
        raise RuntimeError(f"{package}: installed={installed}, loaded={loaded}, expected={version}. "
                           "Перезапустите сеанс Colab и выполните notebook сверху.")

project_env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT), "MPLBACKEND": "Agg"}
smoke_source = textwrap.dedent("""
import importlib, json
names = __MODULES__ + ['src.data', 'src.volatility', 'src.validation', 'src.environment', 'src.portfolio', 'src.environment_data', 'src.portfolio_env', 'src.env_validation', 'src.training_config', 'src.training_data', 'src.regularization', 'src.financial_metrics', 'src.ppo_training', 'src.replay_audit', 'src.pilot_plan', 'src.pilot']
modules = {name: importlib.import_module(name) for name in names}
from src.environment import dependency_closure
result = {
    'module_paths': {n: str(m.__file__) for n, m in modules.items()},
    'dependency_closure': dependency_closure(__ROOTS__),
    'future_stack': {'status': 'not_requested'},
}
if __CHECK_RL__:
    import numpy as np
    import torch
    from packaging.version import Version
    from stable_baselines3 import PPO
    from stable_baselines3.common.env_checker import check_env
    if not (Version('2.3') <= Version(torch.__version__) < Version('3.0')):
        raise RuntimeError('PyTorch does not satisfy SB3 2.7.0')
    if Version(torch.__version__).base_version != '2.11.0':
        raise RuntimeError('Expected preinstalled PyTorch 2.11.0 for the selected 2026.07 baseline')
    array = np.array([1.0, 2.0], dtype=np.float32)
    recovered = torch.from_numpy(array).add(1).numpy()
    if not np.array_equal(recovered, np.array([2.0, 3.0], dtype=np.float32)):
        raise RuntimeError('NumPy/PyTorch CPU interoperability check failed')
    result['future_stack'] = {
        'status': 'passed', 'torch': torch.__version__, 'cuda_build': torch.version.cuda,
        'cuda_available': torch.cuda.is_available(), 'numpy_torch_bridge': 'passed',
        'gymnasium': modules['gymnasium'].__version__,
        'stable_baselines3': modules['stable_baselines3'].__version__,
        'PPO_import': 'passed', 'model_created': False, 'training_performed': False,
        'project_environment_created': False,
    }
print(json.dumps(result, indent=2))
""")
smoke_source = (smoke_source.replace("__MODULES__", repr([module_names.get(p, p) for p in requirements]))
                .replace("__ROOTS__", repr(list(requirements) + (["torch"] if CHECK_RL_COMPATIBILITY else [])))
                .replace("__CHECK_RL__", repr(CHECK_RL_COMPATIBILITY)))
smoke = subprocess.run([sys.executable, "-c", smoke_source], cwd=PROJECT_ROOT,
                       env=project_env, capture_output=True, text=True)
(PREFLIGHT_DIR / "imports.log").write_text(smoke.stdout + "\n" + smoke.stderr, encoding="utf-8")
if smoke.returncode != 0:
    print(smoke.stderr[-4000:])
    raise RuntimeError(f"Ошибка import/ABI preflight; см. {PREFLIGHT_DIR / 'imports.log'}")
smoke_report = json.loads(smoke.stdout)

# Equivalent CLI command: python -m pip check (in this Colab kernel's Python).
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
(PREFLIGHT_DIR / "pip-check.txt").write_text(pip_check.stdout + "\n" + pip_check.stderr, encoding="utf-8")
print(pip_check.stdout or pip_check.stderr)
policy_source = (
    "import json, sys; from src.environment import classify_pip_check; "
    "payload=json.load(sys.stdin); print(json.dumps(classify_pip_check(payload['output'], payload['code'])))"
)
policy = subprocess.run([sys.executable, "-c", policy_source], input=json.dumps({
    "output": pip_check.stdout + "\n" + pip_check.stderr, "code": pip_check.returncode,
}), cwd=PROJECT_ROOT, env=project_env, capture_output=True, text=True, check=True)
classification = json.loads(policy.stdout)
environment = {
    "schema": 2, "profile": "env-v2-colab-2026.07", "rl_check_requested": CHECK_RL_COMPATIBILITY,
    "direct_versions": {p: importlib.metadata.version(p) for p in requirements},
    "versions_before_install": before_install,
    "project_imports": "passed", "subprocess_module_paths": smoke_report["module_paths"],
    "dependency_closure": smoke_report["dependency_closure"],
    "future_stack": smoke_report["future_stack"], "pip_check_policy": classification,
    "pip_check_returncode": pip_check.returncode, "drive_read_write": "passed",
    "python": sys.version, "colab_release_tag": os.environ["COLAB_RELEASE_TAG"],
}
(PREFLIGHT_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Фактические версии:", json.dumps(environment["direct_versions"], indent=2))
print("Dependency closure:", smoke_report["dependency_closure"]["status"])
print("Global pip check:", classification["status"])
print("Future stack:", json.dumps(environment["future_stack"], indent=2))
if smoke_report["dependency_closure"]["status"] != "passed" or classification["status"] == "blocked":
    raise RuntimeError(f"Неустранённый или новый конфликт; изучите {PREFLIGHT_DIR / 'environment.json'}")
print("Project dependency preflight passed; известные предупреждения, если есть, сохранены.")

## 4. План, batch и новая попытка
Здесь ещё нет чтения котировок или обучения. Конфигурации полностью разрешаются до запуска,
показываются список jobs и суммарный бюджет. Plan JSON, Git bundle, код, скиллы, документы,
реальные версии/логи сохраняются на Drive. При продолжении план, Git SHA и device должны совпасть.
После изменения исходников/плана создайте новую batch; старую не переписывайте.

In [ ]:
import shutil
import importlib
import pandas as pd
from IPython.display import display

os.chdir(PROJECT_ROOT)
for name, module in list(sys.modules.items()):
    if name == "src" or name.startswith("src."):
        location = getattr(module, "__file__", None)
        if location and not Path(location).resolve().is_relative_to(PROJECT_ROOT.resolve()):
            raise RuntimeError("src belongs to an older checkout: restart the Colab runtime and run from the top")
sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()
from src.data import sha256_file, write_json
from src.validation import pytest_summary, verify_file_map
from src.training_data import load_development_market
from src.pilot_plan import load_pilot_plan
from src.pilot import prepare_pilot_batch, run_pilot_batch, summarize_pilot_batch, read_pilot_batch

RESOLVED = load_pilot_plan(PROJECT_ROOT / "configs/pilot_plan.json", device=DEVICE)
if RESOLVED["plan"]["stage1_run_id"] != SOURCE_RUN_ID:
    raise ValueError("SOURCE_RUN_ID must match the frozen pilot plan")
BATCH_DIR = prepare_pilot_batch(OUTPUT_ROOT / "batches", RESOLVED, SOURCE_COMMIT, batch_id=BATCH_ID)
ATTEMPT_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
ATTEMPT_DIR = BATCH_DIR / "attempts" / ATTEMPT_ID
ATTEMPT_DIR.mkdir(parents=True, exist_ok=False)
source_paths = sorted(set([
    *PROJECT_ROOT.glob("src/*.py"), *PROJECT_ROOT.glob("tests/test_*.py"),
    *PROJECT_ROOT.glob(".agents/skills/*/SKILL.md"), *PROJECT_ROOT.glob("configs/*.json"),
    *PROJECT_ROOT.glob("docs/*.md"), PROJECT_ROOT / "AGENTS.md",
    PROJECT_ROOT / "requirements-colab.txt", PROJECT_ROOT / "requirements-rl-check.txt",
    PROJECT_ROOT / "notebooks/04_pilot_experiments.ipynb",
]))
source_hashes = {}
for source in source_paths:
    relative = source.relative_to(PROJECT_ROOT)
    target = ATTEMPT_DIR / "provenance" / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    source_hashes[relative.as_posix()] = sha256_file(source)
subprocess.run(["git", "-C", str(REPO_ROOT), "bundle", "create", str(ATTEMPT_DIR / "source.bundle"), "HEAD"], check=True)
for source in PREFLIGHT_DIR.iterdir():
    if source.is_file():
        shutil.copy2(source, ATTEMPT_DIR / source.name)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True)
(ATTEMPT_DIR / "pip-freeze.txt").write_text(freeze.stdout, encoding="utf-8")
attempt_meta = {"stage": 4, "attempt_id": ATTEMPT_ID, "status": "prepared", "git_sha": SOURCE_COMMIT,
    "evaluator_git_sha": SOURCE_COMMIT, "training_git_sha": SOURCE_COMMIT,
    "plan_sha256": RESOLVED["sha256"], "batch_directory": str(BATCH_DIR), "source_sha256": source_hashes,
    "enable_training": ENABLE_TRAINING, "run_tests": RUN_TESTS,
    "tests": {"status": "not_run"}, "test_used": False, "environment": environment}
write_json(ATTEMPT_DIR / "attempt.json", attempt_meta)
display(pd.DataFrame([{k: job[k] for k in ("job_id", "mode", "candidate_id", "seed")} |
    {"lambda_0": job["config"]["lambda_0"], "alpha": job["config"]["alpha"],
     "training_steps": job["config"]["total_timesteps"]} for job in RESOLVED["jobs"]]))
print("Frozen pilot budget:", json.dumps(RESOLVED["budget"], indent=2))
print("Batch ID:", BATCH_DIR.name, "attempt:", ATTEMPT_DIR)
print("Real training enabled:", ENABLE_TRAINING, "full pytest enabled:", RUN_TESTS)

## 5. Полный pytest — до реальных данных и обучения
Для отдельной проверки новой версии можно включить RUN_TESTS=True при ENABLE_TRAINING=False.
Это запускает синтетические тесты, включая маленькие PPO learn/save/load, но не исследовательский пилот.
Failure/error/skipped либо отсутствие ключевых cases блокируют реальные данные и обучение.
Не считать ранее подтверждённые 97 тестов свидетельством прохождения текущей версии.

In [ ]:
if RUN_TESTS:
    if (ATTEMPT_DIR / "test_gate.json").exists():
        raise RuntimeError("This attempt already has immutable test evidence; rerun section 4 to create a new attempt")
    result = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q",
                             f"--junitxml={ATTEMPT_DIR / 'pytest.xml'}"], cwd=PROJECT_ROOT,
                            env={**project_env, "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"}, capture_output=True, text=True)
    (ATTEMPT_DIR / "pytest.log").write_text(result.stdout + "\n" + result.stderr, encoding="utf-8")
    print(result.stdout)
    try:
        if result.returncode:
            raise RuntimeError("Full pytest failed; inspect pytest.log, do not proceed")
        summary = pytest_summary(ATTEMPT_DIR / "pytest.xml")
        import xml.etree.ElementTree as ET
        cases = {c.get("name") for c in ET.parse(ATTEMPT_DIR / "pytest.xml").getroot().iter("testcase")}
        required_cases = {"test_pilot_plan_equal_budgets", "test_pilot_disabled_does_not_train",
                          "test_pilot_resume_preserves_attempts_and_skips_completed",
                          "test_gymnasium_and_sb3_checkers", "test_tiny_synthetic_training_saves_and_reloads"}
        if not required_cases.issubset(cases):
            raise RuntimeError("Required Stage 4/PPO/checker cases were not collected")
        attempt_meta["tests"] = {**summary, "exit_code": 0}
        write_json(ATTEMPT_DIR / "test_gate.json", {"git_sha": SOURCE_COMMIT,
            "tests": attempt_meta["tests"], "pytest_xml_sha256": sha256_file(ATTEMPT_DIR / "pytest.xml")})
    except Exception as error:
        attempt_meta.update(status="tests_failed", error=str(error),
                            tests={"status": "failed", "exit_code": result.returncode})
        write_json(ATTEMPT_DIR / "attempt.json", attempt_meta)
        raise
else:
    attempt_meta["tests"] = {"status": "not_run_preview", "reason": "RUN_TESTS=False; real data/training blocked"}
write_json(ATTEMPT_DIR / "attempt.json", attempt_meta)
print("Actual tests:", attempt_meta["tests"])

## 6. Неизменяемый источник данных, только train/validation
После успешного pytest существующий загрузчик проверит hashes, признаки, календарь и train-only sigma_ref.
Train materialized до 2019, validation до 2022. features_test не открывается; snapshot не скачивается заново.
Без явного RUN_TESTS=True данные не читаются. Чтение train здесь не является обучением.

In [ ]:
train = validation = None
if RUN_TESTS:
    if attempt_meta["tests"].get("status") != "passed":
        raise RuntimeError("Real data require successful full tests")
    train = load_development_market(STAGE1_RUN, "train")
    validation = load_development_market(STAGE1_RUN, "validation")
    for name, data in (("train", train), ("validation", validation)):
        assert data.provenance["snapshot_id"] == RESOLVED["plan"]["snapshot_id"]
        assert data.provenance["test_features_read"] is False
        assert data.adjusted_close.index.max().year <= (2018 if name == "train" else 2021)
        print(name, len(data.features), str(data.features.index.min().date()), str(data.features.index.max().date()))
    attempt_meta["data"] = {"train": train.provenance, "validation": validation.provenance}
else:
    attempt_meta["data"] = "not_loaded_preview"
write_json(ATTEMPT_DIR / "attempt.json", attempt_meta)

## 7. Явно разрешённое обучение либо просмотр плана
Общая процедура train_one_run не меняет PPO/финансовую механику. Каждый job получает собственный
run ID, config, seed, data contract, журналы, final model/normalization и промежуточные checkpoints.
Завершённые jobs при resume проверяются и пропускаются. Ошибка останавливает batch; история
попыток сохраняется. Перезапуск failed/interrupted job — с исходным seed и новым run ID, с полным
бюджетом; дополнительные затраты не скрываются и не считаются равными нулю.

После полного завершения всех 27 jobs выбирается best-within-grid по среднему validation Net Sharpe
трёх seeds, exact tie — по ID. Неполная batch не выбирает победителей. Undefined Sharpe не заменяется нулём.

In [ ]:
try:
    if ENABLE_TRAINING:
        if not RUN_TESTS or attempt_meta["tests"].get("status") != "passed":
            raise RuntimeError("Full tests must pass in this attempt before real training")
        run_pilot_batch(BATCH_DIR, train, validation, ATTEMPT_DIR,
                        enable_training=ENABLE_TRAINING, recover_interrupted=RECOVER_INTERRUPTED)
        attempt_meta["status"] = "training_complete"
    else:
        attempt_meta["status"] = "checks_complete" if RUN_TESTS else "preview_complete"
        print("Real training disabled. No pilot job has been started by this cell.")
    BATCH_REPORT = summarize_pilot_batch(BATCH_DIR)
except BaseException as error:
    attempt_meta.update(status="interrupted" if isinstance(error, (KeyboardInterrupt, SystemExit)) else "failed",
                        error_type=type(error).__name__, error=str(error))
    write_json(ATTEMPT_DIR / "attempt.json", attempt_meta)
    try:
        summarize_pilot_batch(BATCH_DIR)
    except Exception as report_error:
        print("Could not summarize the failed batch:", str(report_error))
    raise
write_json(ATTEMPT_DIR / "attempt.json", attempt_meta)

## 8. Полная сводка, включая pending/failed/interrupted
В CSV присутствуют все предусмотренные jobs. При отсутствии выполнения значения метрик пусты;
это не нулевые/искусственные результаты. Candidate summary требует все seeds; selection требует
всю матрицу. Выбор по validation является разведочным и не доказывает преимущества на независимых данных.

In [ ]:
display(pd.read_csv(BATCH_DIR / "run_summary.csv"))
display(pd.read_csv(BATCH_DIR / "candidate_summary.csv"))
selection = json.loads((BATCH_DIR / "selection.json").read_text(encoding="utf-8"))
print("Selection:", json.dumps(selection, indent=2))
print("Batch status:", BATCH_REPORT["status"], "completed jobs:", BATCH_REPORT["completed_jobs"])
print("All attempts:", BATCH_REPORT["attempts"], "failed:", BATCH_REPORT["failed_attempts"],
      "interrupted:", BATCH_REPORT["interrupted_attempts"], "unknown cost:", BATCH_REPORT["unknown_training_cost_attempts"])

## 9. Итоговая проверка и артефакты
Только фактические статусы: preview/checks не означают завершение обучения.
Не используется test и не рассчитывается значимость. Exact flags старых audits не изменяются.
Сохраните BATCH_ID. Для продолжения после потери runtime откройте тот же SHA, задайте этот BATCH_ID,
RUN_TESTS=True, ENABLE_TRAINING=True; завершённые jobs будут проверены без повторного learn.

Для приёмки предоставьте batch.json, resolved_plan.json, batch_report.json/sha256, все summary/history/selection,
attempt.json/sha256 и provenance/preflight/pytest последней попытки, а также metadata/sidecar,
validation/training metrics и CSV каждого run. Модели и normalization остаются в checkpoint directories.

In [ ]:
state, frozen = read_pilot_batch(BATCH_DIR)
assert frozen["sha256"] == RESOLVED["sha256"] and state["git_sha"] == SOURCE_COMMIT
assert state["test_used"] is False
assert (BATCH_DIR / "batch_report.sha256").read_text().split() == [sha256_file(BATCH_DIR / "batch_report.json"), "batch_report.json"]
verify_file_map(BATCH_DIR, BATCH_REPORT["files_sha256"])
snapshot = ATTEMPT_DIR / "batch_snapshot"
snapshot.mkdir(exist_ok=True)
for name in ("batch.json", "resolved_plan.json", "run_summary.csv", "attempt_history.csv", "candidate_summary.csv",
             "selection.json", "batch_report.json", "batch_report.sha256"):
    shutil.copy2(BATCH_DIR / name, snapshot / name)
verify_file_map(snapshot, BATCH_REPORT["files_sha256"])
assert json.loads((snapshot / "batch_report.json").read_text()) == BATCH_REPORT
assert (snapshot / "batch_report.sha256").read_text().split() == [sha256_file(snapshot / "batch_report.json"), "batch_report.json"]
attempt_meta["batch_report_snapshot"] = "batch_snapshot/batch_report.json"
attempt_meta.update(batch_status=state["status"], completed_at_utc=datetime.now(timezone.utc).isoformat(),
                    batch_report_sha256=sha256_file(BATCH_DIR / "batch_report.json"))
attempt_meta["files_sha256"] = {p.relative_to(ATTEMPT_DIR).as_posix(): sha256_file(p)
    for p in sorted(ATTEMPT_DIR.rglob("*")) if p.is_file() and p.name not in {"attempt.json", "attempt.sha256"}}
write_json(ATTEMPT_DIR / "attempt.json", attempt_meta)
(ATTEMPT_DIR / "attempt.sha256").write_text(sha256_file(ATTEMPT_DIR / "attempt.json") + "  attempt.json\n")
verify_file_map(ATTEMPT_DIR, attempt_meta["files_sha256"])
print("STAGE 4 ATTEMPT:", attempt_meta["status"], "batch:", state["status"])
print("Source/evaluator/training Git SHA:", SOURCE_COMMIT)
print("Full tests actually executed:", attempt_meta["tests"])
print("Scheduled runs / training steps:", frozen["budget"]["runs"], frozen["budget"]["training_steps"])
print("Completed jobs:", BATCH_REPORT["completed_jobs"], "of", frozen["budget"]["runs"])
print("Batch ID:", BATCH_DIR.name)
print("Batch:", BATCH_DIR)
print("Attempt:", ATTEMPT_DIR)
print("No test data used. No statistical superiority conclusion.")